# Лабораторная работа 4: Регрессия
## Вариант: 16 (Прогнозирование стоимости медиа-кампании)
## Используется: California Housing Dataset (sklearn built-in)
### Сложность: Rare

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score)
from scipy import stats

# Настройка визуализации
try:
    plt.style.use('seaborn-v0_8')
except:
    plt.style.use('seaborn')
sns.set_palette("husl")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

warnings.filterwarnings('ignore')

## 1. Загрузка данных

In [ ]:
data = datasets.fetch_california_housing()

X = pd.DataFrame(data["data"], columns=data["feature_names"])
y = data["target"]

df = X.copy()
df['target'] = y

print("=" * 50)
print("Dataset: California Housing")
print("=" * 50)
print(f"\nFeatures ({len(data['feature_names'])}):")
for i, feat in enumerate(data['feature_names'], 1):
    print(f"  {i}. {feat}")
print(f"\nDataset size: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("=" * 50)

## 2. Исследовательский анализ данных (EDA)

In [ ]:
# Статистика
numeric_stats = df.describe(percentiles=[0.25, 0.5, 0.75]).T
print(numeric_stats.to_string())

In [ ]:
# Корреляционная матрица
plt.figure(figsize=(12, 10))
correlation = df.corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('G:/Studies/TSTU/korneeva/4 couse 2 sem/big-data-analysis/labs/lab4/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

target_corr = correlation['target'].abs().sort_values(ascending=False)
print("\nFeature correlation with target:")
for i, (feat, corr) in enumerate(target_corr[:-1].items(), 1):
    print(f"{i:2d}. {feat}: {corr:.4f}")

## 3. Проверка гипотез

In [ ]:
# Гипотеза 1: MedInc положительно коррелирует с target
print("Hypothesis 1: MedInc positively correlates with target")
correlation_medinc = df['MedInc'].corr(df['target'])
t_stat, p_value = stats.pearsonr(df['MedInc'], df['target'])
print(f"  Correlation: {correlation_medinc:.4f}")
print(f"  p-value: {p_value:.2e}")

# Гипотеза 2: Target mean differs for high vs low AveRooms
print("\nHypothesis 2: Target differs for high vs low AveRooms")
median_ave_rooms = df['AveRooms'].median()
high_rooms = df[df['AveRooms'] > median_ave_rooms]['target']
low_rooms = df[df['AveRooms'] <= median_ave_rooms]['target']
print(f"  High: {high_rooms.mean():.4f}, Low: {low_rooms.mean():.4f}")
t_stat, p_value = stats.ttest_ind(high_rooms, low_rooms)
print(f"  p-value: {p_value:.2e}")

## 4. Подготовка данных

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Train target mean: {y_train.mean():.4f}, Test target mean: {y_test.mean():.4f}")

## 5. Построение модели

In [ ]:
def evaluate_regression(model, X_train, X_test, y_train, y_test, model_name):
    """Train model and return metrics"""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    r2 = r2_score(y_test, y_pred)
    
    return {
        'model': model,
        'name': model_name,
        'y_pred': y_pred,
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'mape': mape,
        'r2': r2
    }

# Linear Regression
lr = LinearRegression()
lr_results = evaluate_regression(lr, X_train, X_test, y_train, y_test, "Linear Regression")

print("=" * 50)
print("Linear Regression Results")
print("=" * 50)
print(f"MAE:  {lr_results['mae']:.4f}")
print(f"MSE:  {lr_results['mse']:.4f}")
print(f"RMSE: {lr_results['rmse']:.4f}")
print(f"MAPE: {lr_results['mape']:.2f}%")
print(f"R^2:  {lr_results['r2']:.4f}")

print("\nCoefficients:")
for feat, coef in zip(X.columns, lr.coef_):
    print(f"  {feat}: {coef:.4f}")

## 6. Визуализация результатов

In [ ]:
# Actual vs Predicted
plt.figure(figsize=(10, 6))
plt.scatter(y_test, lr_results['y_pred'], alpha=0.5, edgecolors='black')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Linear Regression: Actual vs Predicted')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('G:/Studies/TSTU/korneeva/4 couse 2 sem/big-data-analysis/labs/lab4/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Residual Plot
residuals = y_test - lr_results['y_pred']
plt.figure(figsize=(10, 6))
plt.scatter(lr_results['y_pred'], residuals, alpha=0.5, edgecolors='black')
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Linear Regression: Residual Plot')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('G:/Studies/TSTU/korneeva/4 couse 2 sem/big-data-analysis/labs/lab4/residual_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr.coef_,
    'Abs_Coefficient': np.abs(lr.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

plt.figure(figsize=(10, 6))
colors = ['red' if x < 0 else 'green' for x in feature_importance['Coefficient']]
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'], color=colors)
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.title('Linear Regression: Feature Importance')
plt.axvline(x=0, color='black', linestyle='-')
plt.tight_layout()
plt.savefig('G:/Studies/TSTU/korneeva/4 couse 2 sem/big-data-analysis/labs/lab4/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Feature Importance:")
for i, row in feature_importance.iterrows():
    print(f"  {row['Abs_Coefficient']:.4f}: {row['Feature']}")

## 7. Итоговые выводы

### Результаты:

1. **Датасет:** California Housing (20,640 образцов, 8 признаков)
   - Целевая переменная: медианная стоимость жилья
   - Пропуски отсутствуют
   - Обнаружены выбросы в AveBedrms, Population

2. **Анализ признаков:**
   - MedInc имеет самую сильную корреляцию с target (0.69)
   - Географические координаты также важны

3. **Гипотезы:**
   - H1: MedInc положительно влияет на стоимость — подтверждено
   - H2: Стоимость различается для разного числа комнат — подтверждено

4. **Модель Linear Regression:**
   - R² = 0.5758 (объясняет ~58% вариации)
   - MAE = 0.5332
   - RMSE = 0.7456

### Вывод:
Линейная регрессия показала умеренный результат (R² ≈ 58%). Для улучшения можно:
- Использовать Ridge/LASSO регуляризацию
- Добавить полиномиальные признаки
- Обработать выбросы